# 15 — Evaluation and Profiling for Clustering (Objective 1)

**Objective 1:** *"To find which operational and infrastructure conditions explain the differences in
shipment weight and reported problems across the 25,000 warehouses"* — by forming performance segments *"followed by
cluster profiling to identify the drivers within each segment"*, evaluated with *"silhouette score, Dunn index and
cluster profiling."*

The model building step produced the segments. This notebook answers three questions it handed on, then answers
Objective 1: how well separated are the final segments; do they differ in the conditions Objective 1 asks about;
and what should each segment be called.

The performance data is a continuum, so separation scores are expected to be modest. Conditions did not define the
segments — they are used here, afterwards, to explain them. Storage issues were excluded from clustering as a
near-duplicate of shipment weight; they are profiled here. Segment 0 is the 908 unrated warehouses assigned by
rule; segments 1–4 come from K-Means. Efficiency is reported from segment totals — problems per 1,000 t and tons
per worker — to avoid the small-denominator distortion that per-warehouse ratios produce. Certificate grades are
presented in their natural order (C < B < B+ < A < A+). Establishment year is analysed on recorded years only,
because missing years were filled at a placeholder value in the data cleaning step. Capacity size and regional zone
are strongly associated with each other and are read together. The low-volume and transport-problem segments hold
across clustering methods; the split of high-volume warehouses into segments 3 and 4 is specific to K-Means.

| § | Question |
|---|---|
| 1 | Do the saved labels, model and data still line up? |
| 2 | How well separated are the final segments — silhouette, Dunn index, Davies–Bouldin, Calinski–Harabasz? |
| 3 | Are those scores low because of the model, or because of the data? A no-structure baseline. |
| 4 | Which segments are coherent, and which merge into their neighbours? |
| 5 | How do the segments differ in performance, in recorded units? |
| 6 | Do the segments differ in the *conditions* Objective 1 asks about? |
| 7 | What should each segment be called? |
| 8 | What separates strong warehouses from weak ones? |

**Input:** `data/preprocessed/warehouse_preprocessed.csv` · `training_and_evaluation/segment_labels.csv` ·
`data/processed/clustering_input_scaled.csv` · `model/kmeans.pkl`
**Output:** tables and figures in `training_and_evaluation/` (listed in §9)

## 0. Setup

**One colour per segment, fixed across every figure in this notebook.** Segment 0 is drawn in neutral grey because it
was assigned by a rule, not found by the clustering. Segments 1–4 take four distinct hues in a fixed order, checked
with a colour-vision-deficiency validator before use. Every figure also labels segments by number, so colour is never
the only way to tell them apart.

Differences above and below a reference are drawn on a blue–grey–red scale, with grey meaning *no difference*.

In [ ]:
import sys, pathlib

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples, davies_bouldin_score, calinski_harabasz_score

paths = obj_paths(1)

SEGMENT_COLOURS = {0: "#898781", 1: "#2a78d6", 2: "#eb6834", 3: "#1baf7a", 4: "#eda100"}
DIVERGING = LinearSegmentedColormap.from_list("below_same_above", ["#2a78d6", "#f0efec", "#e34948"])

---
## 1. Assembling labels, model and data

Three things must agree before anything is evaluated: the labels NB 14 saved, the K-Means model it saved, and the
scaled inputs the model was fitted on. The cell reloads all three, asks the model to label the rated warehouses again,
and joins every label to the full warehouse record — the record profiling needs.

In [ ]:
pre = load_preprocessed()
labels = pd.read_csv(paths["train_eval"] / "segment_labels.csv")
scaled = pd.read_csv(paths["processed"] / "clustering_input_scaled.csv")
saved = joblib.load(paths["model"] / "kmeans.pkl")

inputs = saved["inputs"]
X = scaled[inputs].to_numpy()

# segments of the rated warehouses, in the same row order as X
seg_rated = scaled["Ware_house_ID"].map(labels.set_index("Ware_house_ID")["segment"]).to_numpy()
K = len(np.unique(seg_rated))

predicted = pd.Series(saved["model"].predict(X)).map(saved["segment_of_cluster"]).to_numpy()
assert (predicted == seg_rated).all(), "the saved model no longer reproduces the saved labels"

df = pre.merge(labels, on="Ware_house_ID", how="left", validate="one_to_one")
assert len(df) == 25_000 and df["segment"].notna().all()

print(f"clustering inputs        : {inputs}")
print(f"rated warehouses (X)     : {X.shape[0]:,} x {X.shape[1]}")
print(f"K-Means segments         : {K}")
print(f"saved model reproduces every saved label: True\n")

sizes = df.groupby("segment").agg(warehouses=("Ware_house_ID", "size"), assigned_by=("assigned_by", "first"))
sizes["pct_of_network"] = (100 * sizes["warehouses"] / len(df)).round(2)
sizes

> **Interpretation.**
>
> - Labels, model and data line up. The saved K-Means model relabels all 24,092 rated warehouses exactly
>   as `segment_labels.csv` records them, and every one of the 25,000 warehouses joins to one label. Segment sizes
>   are the ones the model building step saved: **908** (3.63%, by rule), **6,249** (25.00%), **4,018** (16.07%),
>   **6,172** (24.69%) and **7,653** (30.61%).

---
## 2. How well separated are the final segments?

Four internal measures, all computed on **every** rated warehouse. NB 14 used a 10,000-warehouse sample for the
silhouette while scanning nine values of k; for the one segmentation actually chosen, the full computation is
affordable.

| Measure | What it asks | Reading |
|---|---|---|
| **Silhouette** | is each warehouse closer to its own segment than to the next nearest? | −1 to 1, higher better. Rule of thumb (as NB 14): above 0.70 strong, 0.51–0.70 reasonable, 0.26–0.50 weak, 0.25 or below no substantial structure |
| **Dunn index** | smallest distance between two warehouses in *different* segments ÷ largest distance between two warehouses in the *same* segment | higher better; above 1 would mean every segment is narrower than the gap to its nearest neighbour |
| **Davies–Bouldin** | how spread out segments are relative to how far apart they are | lower better |
| **Calinski–Harabasz** | variance between segments relative to variance within them | higher better |

**The Dunn index is not in scikit-learn**, so a short function is defined below. It is computed **exactly**, not on a
sample. That matters for this measure in particular: sampling tends to drop the closest pairs across a boundary (which
raises the numerator) and the most extreme warehouses (which lowers the denominator), so a sampled Dunn index would
flatter the segmentation.

A Dunn index is set by just **two warehouses** — the closest pair across any boundary — so on its own it is hard to read.
The function therefore also returns, for every pair of segments, the two warehouses closest across that boundary, and
the table below translates how far apart they are into **tons and counts** using the unscaled inputs from NB 12.

In [ ]:
base = pd.read_csv(paths["processed"] / "clustering_base.csv")      # the same warehouses, unscaled
assert (base["Ware_house_ID"].to_numpy() == scaled["Ware_house_ID"].to_numpy()).all()

sil = silhouette_samples(X, seg_rated)

# Dunn index = smallest distance between two warehouses in DIFFERENT segments
#            / largest distance between two warehouses in the SAME segment.
# Exact over every warehouse; chunks keep memory small. Also records, for each pair of segments,
# the row numbers of the two warehouses closest across that boundary.
_chunk = 2_000
_segments = np.unique(seg_rated)
diameter = {}
for _s in _segments:
    _A = X[seg_rated == _s]
    diameter[_s] = max(cdist(_A[_i:_i + _chunk], _A).max() for _i in range(0, len(_A), _chunk))
gap, closest_pair = {}, {}
for _i, _a in enumerate(_segments):
    for _b in _segments[_i + 1:]:
        _rows_a, _rows_b = np.flatnonzero(seg_rated == _a), np.flatnonzero(seg_rated == _b)
        gap[(_a, _b)] = np.inf
        for _j in range(0, len(_rows_a), _chunk):
            _d = cdist(X[_rows_a[_j:_j + _chunk]], X[_rows_b])
            _r, _c = np.unravel_index(_d.argmin(), _d.shape)
            if _d[_r, _c] < gap[(_a, _b)]:
                gap[(_a, _b)], closest_pair[(_a, _b)] = _d[_r, _c], (_rows_a[_j + _r], _rows_b[_c])
dunn = min(gap.values()) / max(diameter.values())

metrics = pd.Series({
    "silhouette (mean)": sil.mean(),
    "Dunn index": dunn,
    "Dunn numerator (smallest gap)": min(gap.values()),
    "Dunn denominator (widest segment)": max(diameter.values()),
    "Davies-Bouldin": davies_bouldin_score(X, seg_rated),
    "Calinski-Harabasz": calinski_harabasz_score(X, seg_rated),
}, name="final segments")

print(f"silhouette: every one of the {len(X) * (len(X) - 1) // 2:,} pairs of rated warehouses compared — no sample\n")
print("Dunn denominator — largest distance between two warehouses of the same segment (standardised units):")
print(pd.Series({f"segment {s}": d for s, d in diameter.items()}).round(3).to_string())

rows = []
for (a, b), (p, q) in closest_pair.items():
    differ = (base.loc[p, inputs] - base.loc[q, inputs]).abs()
    rows.append({"segments": f"{a} and {b}", "distance (standardised)": f"{gap[(a, b)]:.4g}",
                 **{f"differ by: {c}": int(differ[c]) for c in inputs}})
print("\nDunn numerator — the two warehouses closest across each boundary, and how they differ in recorded units:")
print(pd.DataFrame(rows).set_index("segments").to_string())
print()
metrics.map("{:,.4g}".format).to_frame()

> **Interpretation.**
>
> - **Silhouette 0.2405, on every pair of warehouses.** It matches the sampled figure from the model building
>   step, so the sample there was representative. It sits below the 0.25 line that the rule of thumb reads as
>   *no substantial structure* — exactly what the continuum expectation recorded before any model was fitted.
>   Davies–Bouldin (1.298) and Calinski–Harabasz (7,661) repeat the model building step's k = 4 row, as they
>   must on the same warehouses and labels.
>
> - **Dunn index 0.0004317 — close to zero, and the closest-pair table shows exactly why.** The numerator is the two
>   nearest warehouses across any boundary; the denominator is the widest segment.
>
> | Boundary | Closest two warehouses differ by | Reading |
> |---|---|---|
> | segment 1 ↔ 4 | **27 t**, and nothing else | the smallest gap — it sets the numerator (0.00238) |
> | segment 1 ↔ 3 | 29 t, and nothing else | |
> | segment 2 ↔ 1, 3, 4 | 939–955 t, and nothing else | same refills, transport issues and breakdowns |
> | segment 3 ↔ 4 | **exactly one refill request**, and nothing else — 0 t | the largest gap of any pair (0.3836) |
>
> - Two warehouses 27 t apart and identical on every count sit in different segments. **The boundary between low and high
>   volume runs straight through the continuous spread of shipment weight**, so near-identical warehouses fall on either
>   side of it. A Dunn index is built from that one closest pair, so any segmentation that cuts a continuous measure scores
>   near zero — the number reports where a boundary runs, not how useful the segments are.
>
> - The boundary between segments 3 and 4 is the opposite case: **no warehouse in either comes closer to the other than a
>   whole refill request.** That split falls cleanly between two refill counts.
>
> - The denominator comes from segment 2, the widest (5.513), ahead of segments 4 (5.448), 3 (5.129) and 1 (4.349).

---
## 3. Low because of the model, or because of the data?

A silhouette of about 0.24 has no meaning on its own. It could mean K-Means has drawn poor boundaries, or that there
are no good boundaries to draw. The expectation recorded in NB 11 §6 predicted the second — this section tests it directly.

The test reuses NB 11's control. **Each input column is shuffled independently**: every column keeps exactly its own
values and spread, but any tendency for particular values to occur *together* — which is what a real group is — is
destroyed. The same K-Means (k = 4, ten starts) is then fitted to the shuffled data and scored the same way. This is
repeated with five different shuffles.

- If the real segments score **clearly better** than the shuffled ones, the clustering has found structure beyond the
  columns' individual spreads.
- If they score **about the same**, the segments are cuts through spread, and a low silhouette is a property of the
  data rather than a fault of the model.

In [ ]:
rows = []
for s in range(5):
    rng = np.random.default_rng(RANDOM_STATE + s)
    X_shuffled = np.column_stack([rng.permutation(X[:, j]) for j in range(X.shape[1])])
    shuffled_labels = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit_predict(X_shuffled)
    # inline Dunn index computation for shuffled data
    _chunk = 2_000
    _segs = np.unique(shuffled_labels)
    _diam = {}
    for _s in _segs:
        _A = X_shuffled[shuffled_labels == _s]
        _diam[_s] = max(cdist(_A[_i:_i + _chunk], _A).max() for _i in range(0, len(_A), _chunk))
    _gap = {}
    for _i, _a in enumerate(_segs):
        for _b in _segs[_i + 1:]:
            _rows_a = np.flatnonzero(shuffled_labels == _a)
            _rows_b = np.flatnonzero(shuffled_labels == _b)
            _gap[(_a, _b)] = np.inf
            for _j in range(0, len(_rows_a), _chunk):
                _d = cdist(X_shuffled[_rows_a[_j:_j + _chunk]], X_shuffled[_rows_b])
                _r, _c = np.unravel_index(_d.argmin(), _d.shape)
                if _d[_r, _c] < _gap[(_a, _b)]:
                    _gap[(_a, _b)] = _d[_r, _c]
    s_dunn = min(_gap.values()) / max(_diam.values())
    s_diameter, s_gap = _diam, _gap
    rows.append({
        "silhouette (mean)": silhouette_samples(X_shuffled, shuffled_labels).mean(),
        "Dunn index": s_dunn,
        "Dunn numerator (smallest gap)": min(s_gap.values()),
        "Dunn denominator (widest segment)": max(s_diameter.values()),
        "Davies-Bouldin": davies_bouldin_score(X_shuffled, shuffled_labels),
        "Calinski-Harabasz": calinski_harabasz_score(X_shuffled, shuffled_labels),
    })
shuffled = pd.DataFrame(rows)

baseline = pd.DataFrame({
    "final segments": metrics,
    "shuffled: mean of 5": shuffled.mean(),
    "shuffled: lowest": shuffled.min(),
    "shuffled: highest": shuffled.max(),
})
baseline["final minus shuffled mean"] = baseline["final segments"] - baseline["shuffled: mean of 5"]
baseline["better when"] = ["higher", "higher", "higher", "lower", "lower", "higher"]
baseline.map(lambda v: f"{v:,.4g}" if isinstance(v, float) else v)

> **Interpretation.**
>
> - The shuffled baseline settles what the low scores mean.
>
> | Measure | Final segments | Shuffled (mean of 5; range) | Verdict |
> |---|---|---|---|
> | silhouette | **0.2405** | 0.2194 (0.2169–0.2212) | better than every shuffle — by 0.021 |
> | Davies–Bouldin | **1.298** | 1.331 (1.325–1.337) | better than every shuffle — by 0.033 |
> | Calinski–Harabasz | **7,661** | 6,508 (6,454–6,558) | better than every shuffle — by 1,153 |
> | Dunn index | 0.0004317 | **0.05183** (0.01362–0.06337) | worse than every shuffle |
>
> - **Three of the four measures say the segments capture real structure** — each beats all five shuffles, which differ
>   little among themselves. **But the margins are small.** A silhouette 0.021 above what K-Means achieves on data with
>   every joint pattern destroyed means most of the modest separation is a property of the data, not a failing of the
>   model. The expectation recorded before modelling is confirmed: these are tiers cut through a continuum, not naturally separate groups.
>
> - **Dunn points the other way, and its two parts explain why.** The shuffled runs' denominators are *wider* (6.108 against
>   5.513), so their higher Dunn comes entirely from the numerator: their closest cross-boundary pairs average 0.317 apart,
>   against 0.00238 for the real segments. The highest shuffled numerator, **0.3836, is exactly the one-refill gap** §2
>   found between segments 3 and 4. In that shuffled run, every pair of warehouses on opposite sides of a boundary was at
>   least a whole refill request apart. On the real data, by contrast, the volume boundary runs through the continuous
>   spread of shipment weight and leaves pairs just 27 t apart (§2). The low Dunn index records that difference in where
>   the boundaries fall. It is not evidence of worse segments, and the three measures that average over every warehouse,
>   rather than resting on one pair, all favour the real segments.


---
## 4. Segment by segment — which segments hold together?

A single mean silhouette can hide one tight segment and one loose one. Each warehouse's own silhouette value says where
it sits: near **+1**, well inside its segment; near **0**, on a boundary; **below 0**, closer on average to another
segment than to its own.

Two summaries per segment: the share of warehouses **below zero** (misplaced, or at least on the wrong side of the
middle), and the share **above 0.5** (clearly inside).

A third view shows *which* segment each one borders. Silhouette itself does not report its nearest other segment, so a
simpler proxy is used and named as such: the **second-nearest K-Means centre** of each warehouse. Its nearest centre is,
by construction, its own segment's — the cell checks that.

In [ ]:
D = saved["model"].transform(X)          # distance from each warehouse to each centre, in the model's own numbering
segment_of_column = np.array([saved["segment_of_cluster"][c] for c in range(D.shape[1])])
closest_first = np.argsort(D, axis=1)
nearest = segment_of_column[closest_first[:, 0]]
next_nearest = segment_of_column[closest_first[:, 1]]
assert (nearest == seg_rated).all(), "every warehouse should sit nearest its own segment's centre"

sep = pd.DataFrame({"segment": seg_rated, "silhouette": sil, "next_nearest_segment": next_nearest})
per_segment = sep.groupby("segment").agg(
    warehouses=("silhouette", "size"),
    mean_silhouette=("silhouette", "mean"),
    median_silhouette=("silhouette", "median"),
    pct_below_0=("silhouette", lambda v: 100 * (v < 0).mean()),
    pct_above_0_5=("silhouette", lambda v: 100 * (v > 0.5).mean()),
)
neighbour_pct = 100 * pd.crosstab(sep["segment"], sep["next_nearest_segment"], normalize="index")
neighbour_pct.columns = [f"next nearest = {c}" for c in neighbour_pct.columns]

print(per_segment.round(4).to_string())
print("\nwhich segment each segment's warehouses sit next to — % of the segment (rows sum to 100):")
neighbour_pct.round(1)

> **Interpretation.**
>
> Reading the summary table: **segment 1 is the most coherent** (mean silhouette 0.2805) with almost no
> misplaced warehouses (0.06% below zero) and the only segment with any warehouses clearly inside (4.24% above
> 0.5). **Segment 2 is the loosest**: 13.22% of its warehouses are closer on average to another segment than to
> their own, and its mean silhouette (0.1939) falls below every other. Segments 3 and 4 sit in between, with
> similarly shaped distributions.
>
> The cross-tabulation shows where each segment's warehouses sit when not near their own centre: **segments 3
> and 4 are each other's nearest neighbour** for about three in five of their warehouses (62.4% and 59.6%),
> reflecting their near-identical performance profiles. **Segment 1** borders segment 4 most often (49.5%), then
> segment 3. **Segment 2** has no single dominant neighbour — it spreads its proximity across all three others.
> The bar chart below shows the full silhouette distribution within each segment.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
y = 0
for s in per_segment.index:
    v = np.sort(sil[seg_rated == s])
    ax.fill_betweenx(np.arange(y, y + len(v)), 0, v, color=SEGMENT_COLOURS[s], linewidth=0)
    ax.text(sil.min() - 0.03, y + len(v) / 2, f"segment {s}", va="center", ha="right", fontsize=9, color="#0b0b0b")
    y += len(v) + 500
ax.axvline(0, color="#c3c2b7", linewidth=1)
ax.axvline(sil.mean(), color="#0b0b0b", linewidth=1)
ax.set_xlim(sil.min() - 0.28, 1)
ax.set_yticks([])
ax.set_xlabel("silhouette value of each warehouse")
ax.set_title("Silhouette of every rated warehouse, sorted within its segment", fontweight="bold")
ax.legend(handles=[Patch(color=SEGMENT_COLOURS[s], label=f"segment {s}") for s in per_segment.index]
          + [Line2D([0], [0], color="#0b0b0b", linewidth=1, label=f"mean, all segments ({sil.mean():.3f})")],
          loc="lower right", frameon=False)
plt.tight_layout()
save_fig(paths["train_eval"] / "silhouette_by_segment.png")
plt.show()

### 5.1 The segments seen in two dimensions

The separation measures give a number but not a picture. Projecting the four scaled performance
measures onto their first two principal components shows what those numbers describe.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X)
var = 100 * pca.explained_variance_ratio_
order = list(per_segment.index)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for s in order:
    m = seg_rated == s
    axes[0].scatter(coords[m, 0], coords[m, 1], s=4, alpha=0.35,
                    color=SEGMENT_COLOURS[s], label=f"segment {s}", linewidth=0)
axes[0].set_title("Rated warehouses in PCA space, coloured by segment", fontweight="bold")
axes[0].legend(markerscale=5, frameon=False, loc="best")

centres = np.vstack([coords[seg_rated == s].mean(axis=0) for s in order])
for s in order:
    m = seg_rated == s
    axes[1].scatter(coords[m, 0], coords[m, 1], s=4, alpha=0.10,
                    color=SEGMENT_COLOURS[s], linewidth=0)
axes[1].scatter(centres[:, 0], centres[:, 1], s=300, marker="X",
                c=[SEGMENT_COLOURS[s] for s in order], edgecolor="#0b0b0b", linewidth=1.5, zorder=5)
for (cx, cy), s in zip(centres, order):
    axes[1].annotate(str(s), (cx, cy), fontsize=11, fontweight="bold",
                     ha="center", va="center", color="white", zorder=6)
axes[1].set_title("Same view with each segment centre marked", fontweight="bold")

for ax in axes:
    ax.set_xlabel(f"PC1 — {var[0]:.1f}% of variance")
    ax.set_ylabel(f"PC2 — {var[1]:.1f}% of variance")

plt.tight_layout()
save_fig(paths["train_eval"] / "segments_in_pca_space.png")
plt.show()

print(f"PC1 {var[0]:.2f}%   PC2 {var[1]:.2f}%   together {var[:2].sum():.2f}%")

### 5.2 Silhouette and the Dunn index against randomised data

Both measures are read against the same segments computed on randomised copies of the data, so that
the question is not "is the score high?" but "is it higher than chance would give?".

In [ ]:
pair = baseline.loc[["silhouette (mean)", "Dunn index"],
                    ["final segments", "shuffled: mean of 5"]]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, name in zip(axes, pair.index):
    vals = [pair.loc[name, "final segments"], pair.loc[name, "shuffled: mean of 5"]]
    bars = ax.bar(["final segments", "randomised data"], vals,
                  color=["#2a78d6", "#c3c2b7"], width=0.55)
    for b, v in zip(bars, vals):
        ax.annotate(f"{v:.5f}", (b.get_x() + b.get_width() / 2, v),
                    ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_title(name, fontweight="bold")
    ax.set_ylim(0, max(vals) * 1.30)
    ax.set_ylabel("higher is better")

plt.tight_layout()
save_fig(paths["train_eval"] / "silhouette_dunn_vs_shuffled.png")
plt.show()

print(pair.round(5).to_string())

> **Interpretation.**
>
> | Segment | Mean silhouette | Below 0 | Above 0.5 | Sits next to (second-nearest centre) |
> |---|---|---|---|---|
> | 1 | **0.2805** — the most coherent | 0.06% | **4.24%** — the only segment with any | 4 (49.5%), 3 (35.7%) |
> | 2 | **0.1939** — the least coherent | **13.22%** | 0% | 1 (41.6%), 4 (35.2%), 3 (23.2%) |
> | 3 | 0.2421 | 0.76% | 0% | **4 (62.4%)** |
> | 4 | 0.2309 | 1.02% | 0% | **3 (59.6%)** |
>
> - Read from the plot: every segment forms a smooth wedge from about zero up to its best-placed warehouses. **No warehouse
>   in any segment reaches much above 0.53** — segment 1's band reaches highest, and the tops of segments 2, 3 and 4 stop
>   a little below 0.5. **Segment 2 is the only segment with a substantial tail below zero**, reaching roughly −0.1;
>   segments 3 and 4 show only hairline slivers there. The bands for segments 3 and 4 have almost the same shape.
>
> - **Segment 1 holds together best**, with almost no misplaced warehouses.
> - **Segment 3 and segment 4 are each other's nearest neighbour** for about three in five of their warehouses — the two
>   high-volume segments border each other more than anything else.
> - **Segment 2 is the loosest.** 13.22% of its warehouses are, on average, closer to another segment than to their own,
>   and it has no single neighbour. It is also the widest segment (5.513, §2): its warehouses share high transport
>   issues but spread across the range of the other measures — §5 shows this directly.
>
> - Segment 2 is also the segment the model building step found **most robust across methods** — Ward and K-Means agree on
>   *which* warehouses belong to it, and those warehouses are spread across the other dimensions.

---
## 5. Performance profile — what each segment looks like, in tons and counts

NB 14 described segments by their **centres**, which cover only the four clustering inputs. The profile below describes
all five segments on all **five** performance measures, including storage issues, in recorded units. Segment 0 is
included, and two reference columns show the rated network and the whole network.

Means and medians are both shown: four of the measures are small counts, where a mean and a median can tell different
stories.

**Efficiency, as NB 13 specified — from segment totals.** Dividing each warehouse's problems by its own tons produced
extreme values for small warehouses (NB 13 §3). Summing first, then dividing, avoids that:

- **problems per 1,000 t** = (total transport issues ÷ 4 + total breakdowns) ÷ total tons × 1,000 — transport issues are
  recorded per year and breakdowns per three months, so transport issues are divided by 4 to put both on a quarterly
  footing, exactly as NB 13 defined the ratio;
- **storage issues per 1,000 t**, the same way;
- **tons per worker** = total tons ÷ total workers. `workers_num` includes the 990 values filled in NB 01.

In [ ]:
performance = ["product_wg_ton", "num_refill_req_l3m", "storage_issue_reported_l3m",
               "transport_issue_l1y", "wh_breakdown_l3m"]

_totals_cols = ["product_wg_ton", "transport_issue_l1y", "wh_breakdown_l3m",
                "storage_issue_reported_l3m", "workers_num"]

spread = df.groupby("segment")[performance].agg(["min", "max"])
ranges = pd.DataFrame({c: spread[c]["min"].astype(str) + " to " + spread[c]["max"].astype(str) for c in performance})
print("range within each segment — lowest to highest warehouse:")
print(ranges.to_string())
print()

# per-segment profile (inlined aggregation logic)
_profile_parts = {}
for _s, _g in df.groupby("segment"):
    _tot = _g[_totals_cols].sum()
    _row = {"warehouses": len(_g), "% of network": 100 * len(_g) / len(df)}
    for _c in performance:
        _row[f"mean {_c}"] = _g[_c].mean()
        _row[f"median {_c}"] = _g[_c].median()
    _row["problems per 1,000 t"] = 1000 * (_tot["transport_issue_l1y"] / 4 + _tot["wh_breakdown_l3m"]) / _tot["product_wg_ton"]
    _row["storage issues per 1,000 t"] = 1000 * _tot["storage_issue_reported_l3m"] / _tot["product_wg_ton"]
    _row["tons per worker"] = _tot["product_wg_ton"] / _tot["workers_num"]
    _profile_parts[f"segment {_s}"] = pd.Series(_row)
perf_profile = pd.DataFrame(_profile_parts)

# rated network
_d = df[df["segment"] > 0]
_tot = _d[_totals_cols].sum()
_row = {"warehouses": len(_d), "% of network": 100 * len(_d) / len(df)}
for _c in performance:
    _row[f"mean {_c}"] = _d[_c].mean()
    _row[f"median {_c}"] = _d[_c].median()
_row["problems per 1,000 t"] = 1000 * (_tot["transport_issue_l1y"] / 4 + _tot["wh_breakdown_l3m"]) / _tot["product_wg_ton"]
_row["storage issues per 1,000 t"] = 1000 * _tot["storage_issue_reported_l3m"] / _tot["product_wg_ton"]
_row["tons per worker"] = _tot["product_wg_ton"] / _tot["workers_num"]
perf_profile["rated network"] = pd.Series(_row)

# whole network
_d = df
_tot = _d[_totals_cols].sum()
_row = {"warehouses": len(_d), "% of network": 100 * len(_d) / len(df)}
for _c in performance:
    _row[f"mean {_c}"] = _d[_c].mean()
    _row[f"median {_c}"] = _d[_c].median()
_row["problems per 1,000 t"] = 1000 * (_tot["transport_issue_l1y"] / 4 + _tot["wh_breakdown_l3m"]) / _tot["product_wg_ton"]
_row["storage issues per 1,000 t"] = 1000 * _tot["storage_issue_reported_l3m"] / _tot["product_wg_ton"]
_row["tons per worker"] = _tot["product_wg_ton"] / _tot["workers_num"]
perf_profile["whole network"] = pd.Series(_row)

perf_profile.round(2)

> **Interpretation.**
>
> Reading the range table and profile together:
>
> - **Segment 0 — the unrated warehouses** have zero storage issues and zero breakdowns for every warehouse
>   (ranges both show 0 to 0). Mean 5,430 t (median 5,090). Refill requests span the full 0–8 range, similar
>   to the rated network.
>
> - **Segment 1 — low volume.** Mean 12,648 t (median 11,089); breakdowns never above 5 and transport issues
>   never above 2. Storage issues and other measures sit near or below the network median. 434.73 t per worker.
>
> - **Segment 2 — transport problems.** Every warehouse in this segment reports 2 to 5 transport issues — no
>   warehouse in segment 1 exceeds 2, and none in segments 3 or 4 exceeds 3. On shipment weight (mean 18,093 t),
>   refills (4.34) and breakdowns (3.67) it sits roughly in the middle. The range in shipment weight is the
>   widest of any segment (4,055 to 48,142 t). **0.24 problems per 1,000 t** — the highest of any rated segment.
>
> - **Segments 3 and 4 — high volume.** Mean 28,354 t and 28,864 t. Storage issues, transport issues, breakdowns
>   and efficiency (0.15 problems per 1,000 t each) are near-identical. **Refill requests separate them cleanly**:
>   segment 3 spans 0 to 3, segment 4 spans 4 to 8, with no overlap. 978–1,001 t per worker. The five boxplots
>   below display these distributions.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(21, 4.8))
for ax, c in zip(axes, performance):
    sns.boxplot(data=df, x="segment", y=c, hue="segment", palette=SEGMENT_COLOURS, legend=False,
                linewidth=1, fliersize=1.5, ax=ax)
    ax.set_title(c, fontsize=10)
    ax.set_xlabel("segment"); ax.set_ylabel("")
fig.suptitle("Performance measures by segment  (segment 0 = unrated, assigned by rule)", fontweight="bold", y=1.03)
plt.tight_layout()
save_fig(paths["train_eval"] / "performance_by_segment.png")
plt.show()

> **Interpretation.**
>
> - Read from the tables and checked against the boxplots:
>   - **Segment 0 — the unrated warehouses.** Mean 5,430 t (range 2,065–14,149), and **zero storage issues and zero breakdowns
>     for every warehouse**. Their refill requests look like everyone else's (mean 3.98, range 0–8). 187 t per worker —
>     far below any rated segment.
>
> - **Segment 1 — lowest volume among rated warehouses, fewest problems per warehouse.** Mean 12,648 t (median 11,089).
>   Breakdowns 2.07 on average and never more than 5, transport issues never more than 2. 434.73 t per worker.
>
> - **Segment 2 — defined by transport problems.** **Every warehouse reports 2 to 5 transport issues** (mean 3.07). No
>   warehouse in segment 1 reports more than 2, and none in segments 3 or 4 more than 3. On everything else segment 2 sits in
>   the middle — mean 18,093 t, 3.67 breakdowns, refills like the network (4.34). It holds warehouses of almost every size,
>   **4,055–48,142 t**. Read from the boxplot, its middle half spans the tallest band of shipment weight of any segment —
>   roughly 11,000 to 25,000 t. That is consistent with it being the widest segment in §2 and the least coherent in §4: it
>   collects warehouses of very different volumes that share one problem.
>
> - **Segments 3 and 4 — high volume, alike in everything but refills.** In the boxplots their boxes for shipment weight,
>   storage issues, transport issues and breakdowns are near-identical. The numbers agree: mean 28,354 t against
>   28,864 t, storage issues 22.15 against 22.39, breakdowns 4.25 against 4.33, 0.15 problems per 1,000 t each, and
>   978–1,001 t per worker. The one panel where they separate is refills, and there **they do not overlap at all: segment
>   3 spans 0 to 3 requests, segment 4 spans 4 to 8** — the clean split §2 found.
>
> - **Three readings that cut across segments:**
>
> - **Volume tiers overlap heavily at the edges.** Segment 1 reaches 32,130 t, while segments 3 and 4 go down to 5,061 t and
>   4,093 t. Segments are combinations of measures, not bands of shipment weight.
> - **Storage issues grow in exact step with volume.** Per 1,000 t they are 0.78 in segments 1, 3 and 4 and 0.80 in
>   segment 2. In absolute terms high-volume warehouses report more than twice segment 1's storage issues; per ton, no
>   segment is better or worse. This is the near-duplicate relationship set aside in the feature selection step, now visible
>   in segment totals.
> - **More breakdowns, but fewer per ton.** The high-volume segments break down more often than segment 1 (4.25–4.33
>   against 2.07), yet record the fewest problems per 1,000 t (0.15 against 0.17). Segment 2 records the most, **0.24**.
>
> - *A note on numbers.* These are means of the warehouses assigned to each segment, so they differ slightly from the
>   model centres the model building step reported. The centres are the model's reference points; the profile describes
>   the warehouses.

---
## 6. Do the segments differ in their conditions?

This is the question Objective 1 actually asks: which *conditions* explain the differences in performance.

**Tests run on segments 1–4 only (24,092 warehouses).** Segment 0 was defined by the certificate, so including it would
guarantee a certificate association by construction — the EDA found the unrated flag and the certificate perfectly
associated (V = 1.00). Segment 0 is still described alongside the others in §6.2, but it cannot be *tested* against a
condition it was built from.

**Two tests per condition, with an effect size that decides.** With 24,092 warehouses almost any
difference is statistically significant, so — as in the earlier analysis — p-values say whether a difference exists and
the effect size says whether it matters.

| Condition type | Tests | Effect size | Rule of thumb — small / medium / large |
|---|---|---|---|
| numeric | one-way ANOVA; Kruskal–Wallis (makes no assumption about shape, which matters for counts and flat columns) | **η²** — share of the condition's variance explained by segment | 0.01 / 0.06 / 0.14 |
| 0/1 and categorical | chi-square | **Cramér's V** | 0.10 / 0.30 / 0.50 **divided by √(m − 1)**, where m is the smaller of the table's two dimensions — Cohen's thresholds, adjusted because V runs lower for larger tables |

The last column of the table turns the effect into business terms: for a numeric condition, the range of segment
medians; for a categorical one, the level whose share differs most across segments.

`wh_est_year` uses **recorded years only**: missing years were filled at a placeholder value in the data cleaning
step, which would pull every segment toward the same value.

In [ ]:
rated = df[df["segment"] > 0]

conditions_numeric = ["wh_est_year", "workers_num", "dist_from_hub", "Competitor_in_mkt",
                      "retail_shop_num", "distributor_num", "govt_check_l3m"]
conditions_binary = ["electric_supply", "temp_reg_mach", "flood_proof", "flood_impacted"]
conditions_categorical = ["approved_wh_govt_certificate", "Location_type", "WH_capacity_size",
                          "WH_regional_zone", "zone", "wh_owner_type"]

rows = []
for c in conditions_numeric:
    d = rated[rated["wh_est_year_missing"] == 0] if c == "wh_est_year" else rated
    groups = [v.to_numpy() for _, v in d.groupby("segment")[c]]
    # eta squared — share of variance in c explained by segment membership
    _values, _groups = d[c], d["segment"]
    _grand_mean = _values.mean()
    _between = sum(len(_v) * (_v.mean() - _grand_mean) ** 2 for _, _v in _values.groupby(_groups))
    effect = _between / ((_values - _grand_mean) ** 2).sum()
    medians = d.groupby("segment")[c].median()
    rows.append({
        "condition": c, "kind": "numeric", "warehouses_used": len(d),
        "p (ANOVA or chi-square)": stats.f_oneway(*groups).pvalue,
        "p (Kruskal-Wallis)": stats.kruskal(*groups).pvalue,
        "effect size": "eta squared", "effect": effect,
        "size": "large" if effect >= 0.14 else "medium" if effect >= 0.06 else "small" if effect >= 0.01 else "negligible",
        "largest difference across segments 1-4": f"median {medians.min():g} to {medians.max():g}",
    })

for c in conditions_binary + conditions_categorical:
    table = pd.crosstab(rated["segment"], rated[c])
    chi2, p = stats.chi2_contingency(table, correction=False)[:2]
    m = min(table.shape)
    effect = np.sqrt(chi2 / (table.to_numpy().sum() * (m - 1)))
    shares = 100 * table.div(table.sum(axis=1), axis=0)
    level = (shares.max() - shares.min()).idxmax()
    rows.append({
        "condition": c, "kind": "0/1" if c in conditions_binary else "categorical", "warehouses_used": len(rated),
        "p (ANOVA or chi-square)": p, "p (Kruskal-Wallis)": np.nan,
        "effect size": "Cramér's V", "effect": effect,
        "size": "large" if effect >= 0.50 / np.sqrt(m - 1) else "medium" if effect >= 0.30 / np.sqrt(m - 1) else "small" if effect >= 0.10 / np.sqrt(m - 1) else "negligible",
        "largest difference across segments 1-4": f"{c} = {level}: {shares[level].min():.1f}% to {shares[level].max():.1f}%",
    })

order = {"large": 0, "medium": 1, "small": 2, "negligible": 3}
tests = (pd.DataFrame(rows).assign(rank=lambda t: t["size"].map(order))
         .sort_values(["rank", "effect"], ascending=[True, False]).drop(columns="rank").reset_index(drop=True))

print(f"conditions tested: {len(tests)}   —   by size: {tests['size'].value_counts().to_dict()}\n")
tests.assign(**{
    "p (ANOVA or chi-square)": tests["p (ANOVA or chi-square)"].map("{:.3g}".format),
    "p (Kruskal-Wallis)": tests["p (Kruskal-Wallis)"].map(lambda v: "—" if pd.isna(v) else f"{v:.3g}"),
    "effect": tests["effect"].round(4),
})

> **Interpretation.**
>
> - Of 17 conditions, **one differs strongly between segments, two differ slightly, and fourteen do not
>   differ in any meaningful way.**
>
> | Size | Condition | Effect | In business terms |
> |---|---|---|---|
> | **large** | establishment year | η² **0.3658** | segment medians from 2005 to 2017 — *on recorded years only; see §6.3* |
> | small | temperature regulation | V 0.1997 | share with regulation from 17.8% to 42.2% |
> | small | certificate grade | V 0.0821 | share graded A+ from 10.7% to 21.1% |
> | negligible | the other 14 | V ≤ 0.0510, η² ≤ 0.0005 | differences of a few percentage points or less, or identical medians |
>
> - **Statistical significance is not what decides.** `Location_type` is significant at p = 1.56e-13, yet its effect is
>   negligible (V 0.0510) — its rural share only moves between 90.5% and 93.9%. `distributor_num` is significant by both
>   tests (p ≈ 0.009) with η² of 0.0005 and segment medians of 41 to 43. At 24,092 warehouses small differences reach
>   significance easily. Ownership, zone, regional zone, capacity size, electric back-up, both flood indicators and the
>   numeric conditions other than distributors and distance are **not significant even at 5%** (p from 0.0628 to 0.917).
>
> - This is the same short list the EDA found related to *performance*: age, certificate and temperature regulation.
>   The segments add a sharper view of *which* segments differ, in §6.2 and §6.3. Capacity size and regional zone, which
>   the EDA found strongly associated with each other, are both negligible here — so reading them together adds nothing.

### 6.2 How the shares of each condition differ by segment

The heatmap shows, for every level of every 0/1 and categorical condition, how far each segment's share sits **above or
below the rated network's share**, in percentage points. For 0/1 conditions only the share with the condition (`= 1`)
is shown. Segment 0 is included for description. Its certificate rows differ by construction — every segment 0
warehouse is `Unrated` — so the colour scale is capped, and every cell carries its exact figure.

The table printed beneath gives the shares themselves.

In [ ]:
level_order = {"approved_wh_govt_certificate": ["C", "B", "B+", "A", "A+", "Unrated"],
               "WH_capacity_size": ["Small", "Mid", "Large"]}

share_tables = []
for c in conditions_binary + conditions_categorical:
    t = 100 * pd.crosstab(df[c], df["segment"], normalize="columns")
    t["rated network"] = 100 * rated[c].value_counts(normalize=True)
    t["whole network"] = 100 * df[c].value_counts(normalize=True)
    t = t.fillna(0)
    if c in conditions_binary:
        t = t.loc[[1]]
    if c in level_order:
        t = t.reindex(level_order[c])
    t.index = [f"{c} = {level}" for level in t.index]
    share_tables.append(t)

shares = pd.concat(share_tables)
shares.columns = [f"segment {c}" if not isinstance(c, str) else c for c in shares.columns]
segment_cols = [f"segment {s}" for s in range(K + 1)]
difference = shares[segment_cols].sub(shares["rated network"], axis=0).round(1) + 0.0    # + 0.0 turns -0.0 into 0.0

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(difference, annot=True, fmt="+.1f", cmap=DIVERGING, center=0, vmin=-20, vmax=20,
            linewidths=2, linecolor="white", annot_kws={"size": 8},
            cbar_kws={"label": "percentage points above (+) or below (−) the rated network's share"}, ax=ax)
ax.set_title("Where each segment's share of a condition differs from the rated network\n(colour capped at ±20 pp; "
             "cell figures are exact)", fontweight="bold")
ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout()
save_fig(paths["train_eval"] / "condition_differences_by_segment.png")
plt.show()

print("share of warehouses with each condition level (%) — each column sums to 100 within a condition:")
print(shares.round(1).to_string())

> **Interpretation.**
>
> - Read from the heatmap and confirmed against the table beneath it:
>   - **Among segments 1–4, only two conditions show visible colour.**
>
> - **Temperature regulation separates the two high-volume segments.** Segment 4 is **42.2%** regulated (+10.8 pp) and
>   segment 3 only **17.8%** (−13.6 pp); segments 1 and 2 sit near the network (30.0% and 34.0%). Segments 3 and 4 are
>   alike on every performance measure except refills (§5), so the one condition separating them lines up with the one
>   measure that does. The EDA found temperature regulation related to refill requests independently of the unrated
>   group. Frequently refilled, high-volume warehouses are more often temperature regulated — as association, not cause.
> - **Certificate grades improve step by step with volume.** The share graded C falls from 29.4% (segment 1) through
>   23.4% and 20.2% to 19.4% (segment 4). The share graded A+ rises from 10.7% through 16.2% and 20.3% to 21.1%.
>   Segment 1 has the most C grades (+6.5 pp) and the fewest A+ (−6.7 pp). The gradient is consistent but small.
> - **Location type** moves slightly in segment 1, with 6.1% urban against 9.0–9.5% elsewhere (−2.4 pp) — negligible in
>   effect (§6.1).
>
> - **Everything else is flat.** Every cell for capacity size, regional zone, zone, ownership, electric back-up and both
>   flood indicators in segments 1–4 lies within ±1.0 pp of the rated network.
>
> - **Segment 0 (described, not tested).** Every one of its warehouses is `Unrated`, by construction. It stands apart on
>   two further conditions: only **1.3%** are temperature regulated (−30.1 pp), and **all of them are rural** (+8.5 pp).
>   On capacity, zone and ownership it stays within 2.5 pp of the rated network.

### 6.3 Numeric conditions, and the recorded-year caveat

Medians for every numeric condition by segment. `wh_est_year` is on recorded years only, so the **share of each segment
with a recorded year** is printed with it: a segment's year profile is only as representative as that share. Earlier
data cleaning found the year missing for *every* warehouse with 0–2 refill requests, and refill frequency is one of
the clustering inputs — so recorded-year shares may differ by segment, and this cell checks rather than assumes.

In [ ]:
numeric_profile = {}
for c in conditions_numeric:
    d = df[df["wh_est_year_missing"] == 0] if c == "wh_est_year" else df
    medians = d.groupby("segment")[c].median()
    medians.index = [f"segment {s}" for s in medians.index]
    medians["rated network"] = d.loc[d["segment"] > 0, c].median()
    medians["whole network"] = d[c].median()
    numeric_profile[f"median {c}" + ("  (recorded years only)" if c == "wh_est_year" else "")] = medians

recorded = 100 * (1 - df.groupby("segment")["wh_est_year_missing"].mean())
recorded.index = [f"segment {s}" for s in recorded.index]
recorded["rated network"] = 100 * (1 - rated["wh_est_year_missing"].mean())
recorded["whole network"] = 100 * (1 - df["wh_est_year_missing"].mean())
numeric_profile["% with a recorded establishment year"] = recorded

numeric_profile = pd.DataFrame(numeric_profile).T
print(numeric_profile.round(1).to_string())

years = df[df["wh_est_year_missing"] == 0]
counts = years.groupby("segment").size()

print("\nrefill requests and transport issues of the warehouses WITH a recorded year only:")
print(years.groupby("segment")[["num_refill_req_l3m", "transport_issue_l1y"]].agg(["min", "median", "max"]).to_string())

fig, ax = plt.subplots(figsize=(10, 4.8))
sns.boxplot(data=years, x="segment", y="wh_est_year", hue="segment", palette=SEGMENT_COLOURS, legend=False,
            linewidth=1, fliersize=1.5, ax=ax)
ax.set_xticks(range(len(counts)))
ax.set_xticklabels([f"segment {s}\n{n:,} recorded" for s, n in counts.items()])
ax.set_xlabel(""); ax.set_ylabel("establishment year")
ax.set_title("Establishment year by segment — recorded years only", fontweight="bold")
plt.tight_layout()
save_fig(paths["train_eval"] / "establishment_year_by_segment.png")
plt.show()

> **Interpretation.**
>
> - **Six of the seven numeric conditions are the same in every segment.** Median workers are 28 in all five segments and
>   median competitors 3 in all five. Distance from hub runs 162–167, retail shops 4,849–4,887, distributors 41–43 and
>   government checks 19–21. None of these separates any segment from any other.
>
> - **Establishment year does differ — and falls as volume rises.** Read from the boxplot: segment 0's warehouses are the
>   newest (box 2021–2023, median 2022). Segment 1 is next (box 2013–2020, median 2017), then segment 2 (2004–2015, median
>   2009). **Segments 3 and 4 have identical boxes, 2001–2010, both with median 2005.** The higher the volume tier, the
>   older the warehouses. The two high-volume segments are the same age, so age does not explain the split between them.
>
> - **Caveat — which warehouses these years describe.**
>   - **Recorded shares differ sharply by segment:** 87.2% in segment 4 and 57.8% in segment 1, but only **25.4%** in
>     segment 2 and **21.6%** in segment 3.
>   - **No warehouse with a recorded year has fewer than three refill requests, in any segment.** The minimum is 3 in
>     segments 0–3, and 4 in segment 4, whose refills start at 4. None has five transport issues (maximum 4). This is the
>     pattern earlier data cleaning found in the whole file.
>   - So **segment 3's year profile describes only 1,335 warehouses, all with exactly three refill requests** (minimum,
>     median and maximum all 3). Those are the least typical members of a segment whose median is one request.
>
> - Earlier data cleaning found the median year 2009 or 2010 at every refill level where years *are* recorded, so there
>   is no sign that refill frequency shifts the year. But for warehouses with 0–2 refill requests the year is never
>   recorded, and their age is simply unknown. The age comparison is therefore **most reliable between segment 1 (2017)
>   and segment 4 (2005)**, which rest on 57.8% and 87.2% of their warehouses. For segments 2 and 3 it describes a
>   minority. The η² of 0.3658 in §6.1 inherits the same caveat.

---
## 7. Naming the segments

A segment name must pass three checks:

- **Truth:** it must describe the warehouses in the segment (§5).
- **Difference:** it must say what sets the segment apart from the others.
- **Restraint:** it must not claim more than the evidence supports, especially for the split between segments 3 and 4.

> **Decision — segment naming.**
>
> | Segment | Name | Why this name is true (§) | Robust to method |
> |---|---|---|---|
> | 0 | **Newly commissioned, not yet rated** | every warehouse `Unrated`; zero storage issues and breakdowns; median year 2022; mean 5,430 t (§5, §6) | by definition — assigned by rule |
> | 1 | **Low volume, few breakdowns** | lowest rated volume (mean 12,648 t); fewest breakdowns (2.07, never above 5); transport issues never above 2; youngest rated segment (median 2017) (§5, §6.3) | largely — 76% kept together by Ward |
> | 2 | **Transport-problem** | every warehouse has 2–5 transport issues, and no warehouse elsewhere more than 3; most problems per 1,000 t (0.24); every size of warehouse (§5) | yes — 86% kept together by Ward |
> | 3 | **High volume — rarely refilled** | mean 28,354 t; refills 0–3 only; 17.8% temperature regulated (§5, §6.2) | no — the 3/4 split is specific to K-Means |
> | 4 | **High volume — frequently refilled** | mean 28,864 t; refills 4–8 only; 42.2% temperature regulated (§5, §6.2) | no — the 3/4 split is specific to K-Means |
>
> **How segments 3 and 4 are reported.**
>
> - Report them as two sub-segments of one high-volume tier.
> - Put the high-volume tier first in every summary.
> - Put the refill split second.
> - Carry the caveat that the split is K-Means-specific.
>
> **Why the split is kept.**
>
> - Refills split cleanly: 0–3 against 4–8.
> - No warehouse crosses the split by less than one whole refill request (§2).
> - Temperature regulation differs meaningfully: 17.8% against 42.2% (§6.2).
> - K-Means reproduces the four rated segments across starts (mean ARI 0.9997).
>
> **Why the split is not overstated.**
>
> - Segments 3 and 4 are almost identical on volume.
> - They are almost identical on storage issues, transport issues, breakdowns and efficiency.
> - Both have median establishment year 2005.
> - They are each other's nearest neighbour (§4).
> - Ward does not reproduce the split (ARI 0.33–0.55 across five samples).
>
> **Rejected.**
>
> - Merge 3 and 4 completely: this would discard a clean split aligned with temperature regulation.
> - Present 3 and 4 as fully independent segment types: this would overstate the evidence.
> - Use evaluative names such as strong or weak: strength depends on whether the denominator is warehouse count or tons shipped.
>
> Names describe what segments **are**. Section 8 addresses which segments are strong or weak.

In [ ]:
segment_names = pd.DataFrame.from_dict({
    0: {'name': 'Newly commissioned, not yet rated', 'tier': 'Unrated', 'defined_by': 'rule: certificate Unrated', 'robust_to_method': 'by definition'},
    1: {'name': 'Low volume, few breakdowns', 'tier': 'Low volume', 'defined_by': 'K-Means', 'robust_to_method': 'largely (76% kept together by Ward)'},
    2: {'name': 'Transport-problem', 'tier': 'Transport problems', 'defined_by': 'K-Means', 'robust_to_method': 'yes (86% kept together by Ward)'},
    3: {'name': 'High volume — rarely refilled', 'tier': 'High volume', 'defined_by': 'K-Means', 'robust_to_method': 'tier yes; split from segment 4 is K-Means-specific'},
    4: {'name': 'High volume — frequently refilled', 'tier': 'High volume', 'defined_by': 'K-Means', 'robust_to_method': 'tier yes; split from segment 3 is K-Means-specific'},
}, orient="index")
segment_names.index.name = "segment"
segment_names.insert(0, "warehouses", df.groupby("segment").size())
segment_names.insert(1, "% of network", (100 * segment_names["warehouses"] / len(df)).round(2))
segment_names

> **Interpretation.**
>
> The table confirms the naming decision. Segments 0 through 2 have names tied to the one characteristic that
> defines them — assignment rule, volume tier, and problem type respectively. Segments 3 and 4 share the same
> tier label (high volume) with a sub-description (refill frequency) that acknowledges the split's specificity
> to K-Means. The `robust_to_method` column records which findings from the hierarchical comparison held up: the
> low-volume and transport-problem segments were largely reproduced by Ward linkage, while the two high-volume
> sub-segments were not.

---
## 8. Answering Objective 1 — what separates strong warehouses from weak ones?

> **The answer, in business terms.**
>
> **1. "Strong" depends on how performance is measured, so both views are stated.**
>
> | View | Best | Worst |
> |---|---|---|
> | **per warehouse** — how much it ships | high volume (segments 3 and 4): 28,354 t and 28,864 t, more than twice segment 1 | low volume (segment 1): 12,648 t |
> | **per warehouse** — breakdowns | low volume (segment 1): 2.07, never more than 5 | high volume (segments 3 and 4): 4.25 and 4.33 |
> | **per warehouse** — transport issues | high volume and low volume: at most 2 or 3 per warehouse | transport-problem (segment 2): every warehouse 2–5 |
> | **per ton shipped** — problems per 1,000 t (efficiency measure from the feature engineering step) | high volume (segments 3 and 4): **0.15** | transport-problem (segment 2): **0.24** |
> | **per worker** — tons per worker | high volume: 978–1,001 t | low volume: 434.73 t |
>
> Counting problems per warehouse gives a split picture: low-volume warehouses break down least, high-volume ones most.
> **Measured by efficiency — output relative to problems and staff — the high-volume tier is the strongest part of the
> network and the transport-problem segment the weakest.** High-volume warehouses log more breakdowns and storage issues
> than low-volume ones, but in proportion to what they ship they log fewer problems (§5). Storage issues rise in exact
> step with volume (0.78–0.80 per 1,000 t in every rated segment), so they mark size rather than weakness.
>
> The **newly commissioned** warehouses (segment 0) are not ranked. They ship the least (5,430 t) and log no storage
> issues or breakdowns, which describes warehouses not yet fully operating rather than warehouses performing well.
>
> **2. Which conditions go with the differences.** Three, of the seventeen recorded:
>
> - **Warehouse age — the clearest.** The higher the volume tier, the older the warehouses: median establishment year
>   2017 for low volume, 2009 for transport-problem, 2005 for both high-volume sub-segments (η² 0.3658). *Caveat:* this
>   rests on recorded years only, which exist only for warehouses with three or more refill requests (§6.3).
> - **Certificate grade — small but consistent.** The share graded A+ rises from 10.7% (low volume) to 21.1% (high
>   volume, frequently refilled); the share graded C falls from 29.4% to 19.4%.
> - **Temperature regulation — the one condition that separates the two high-volume sub-segments.** 42.2% of frequently
>   refilled warehouses are regulated, against 17.8% of rarely refilled ones.
>
> **Nothing else separates the segments.** Location, zone, regional zone, capacity size, ownership, electric back-up,
> flood exposure, staffing, distance from hub, competitors, retail shops, distributors and government checks are all
> within about a percentage point, or have the same median, in every rated segment.
>
> **3. The weakest segment is not explained by any recorded condition.** The transport-problem segment matches the
> network on every condition. Its temperature-regulation share is 34.0% against 31.4%, and its certificate shares are
> within 1.2 pp. Every other categorical and 0/1 condition is within 1.0 pp. Its median year is 2009, the network median,
> and its other numeric medians are practically the network's — 28 workers in both, 167 against 164 from the hub, 4,851
> against 4,858.5 retail shops. **Whatever drives
> transport problems is not in this dataset.** For the business, this is the most practical finding: the segment that most
> needs attention cannot be targeted using the recorded infrastructure, location or ownership fields.
>
> **4. Against the stated project expectation.** *"Shipment weight is driven mainly by reported storage issues and
> warehouse age, while location and ownership appear weak"* — **supported.** Volume tiers align with age. Storage issues
> move in lockstep with volume. Location (V 0.0510) and ownership (V 0.0155) are negligible.
>
> **5. Limits on these conclusions.**
>
> - **The segments are tiers through a continuum, not natural groups.** Silhouette 0.2405, only 0.021 above structureless
>   data (§3). Warehouses near a boundary — some just 27 t apart (§2) — could reasonably sit on either side.
> - **The split of the high-volume tier into rarely and frequently refilled is specific to K-Means.**
> - **Age findings describe warehouses with a recorded year:** 52.5% of the network, and only those with three or more
>   refill requests.
> - **The data is a single snapshot, so every finding is association, not cause.** It cannot show whether older warehouses
>   ship more *because* of their age, whether temperature regulation *leads to* more refill requests, or whether both
>   follow from something not recorded here.

---
## 9. Save

In [ ]:
_ = save_table(baseline.drop(columns="better when"), paths["train_eval"] / "evaluation_metrics.csv")
_ = save_table(per_segment.join(neighbour_pct), paths["train_eval"] / "segment_separation.csv")
_ = save_table(perf_profile, paths["train_eval"] / "segment_performance_profile.csv")
_ = save_table(tests, paths["train_eval"] / "segment_condition_tests.csv", index=False)
_ = save_table(shares, paths["train_eval"] / "segment_condition_shares.csv")
_ = save_table(numeric_profile, paths["train_eval"] / "segment_numeric_conditions.csv")
_ = save_table(segment_names, paths["train_eval"] / "segment_names.csv")

---
## 10. Checks

The saved tables are read back and checked for completeness: every segment present, every warehouse counted once, every
condition tested, and every categorical condition's shares summing to 100% within each segment.

In [ ]:
back = pd.read_csv(paths["train_eval"] / "segment_performance_profile.csv", index_col=0)
assert back.loc["warehouses", segment_cols].sum() == 25_000
assert back.loc["warehouses", "rated network"] == 24_092

back = pd.read_csv(paths["train_eval"] / "segment_separation.csv", index_col=0)
assert back["warehouses"].sum() == 24_092 and list(back.index) == list(range(1, K + 1))

back = pd.read_csv(paths["train_eval"] / "segment_condition_tests.csv")
assert len(back) == len(conditions_numeric) + len(conditions_binary) + len(conditions_categorical) == 17
assert back["condition"].is_unique

back = pd.read_csv(paths["train_eval"] / "segment_condition_shares.csv", index_col=0)
for c in conditions_categorical:
    sums = back.loc[back.index.str.startswith(c + " = "), segment_cols].sum()
    assert np.allclose(sums, 100), (c, sums)

back = pd.read_csv(paths["train_eval"] / "evaluation_metrics.csv", index_col=0)
assert back.notna().all().all() and len(back) == 6
back = pd.read_csv(paths["train_eval"] / "segment_names.csv", index_col=0)
assert list(back.index) == list(range(K + 1)) and back["warehouses"].sum() == 25_000
assert back["name"].notna().all() and back["name"].is_unique

print("all checks passed")

---
## Summary

**Evaluation — the required measures, on every rated warehouse.**

| Measure | Final segments | Shuffled baseline (mean of 5) | Reading |
|---|---|---|---|
| Silhouette | 0.2405 | 0.2194 | weak separation, as expected from the data forming a continuum; slightly better than structureless data |
| Dunn index | 0.0004317 | 0.05183 | near zero because the volume boundary cuts the continuous shipment spread (closest pair 27 t apart) |
| Davies–Bouldin | 1.298 | 1.331 | slightly better than structureless data |
| Calinski–Harabasz | 7,661 | 6,508 | better than structureless data |

The segments capture real but modest structure: **tiers cut through a continuum**, not naturally separate groups. Segment
1 is the most coherent (silhouette 0.2805). Segment 2 is the loosest (0.1939, 13.22% below zero) yet the most robust across
methods. Segments 3 and 4 split cleanly at a single refill request.

**Profiling — the segments, named.**

| Segment | Name | Share | Key facts |
|---|---|---|---|
| 0 | Newly commissioned, not yet rated | 3.63% | 5,430 t; no storage issues or breakdowns; median year 2022; 1.3% temperature regulated |
| 1 | Low volume, few breakdowns | 25.00% | 12,648 t; 2.07 breakdowns; youngest rated (2017); most C grades |
| 2 | Transport-problem | 16.07% | every warehouse 2–5 transport issues; **0.24 problems per 1,000 t**; matches the network on every condition |
| 3 | High volume — rarely refilled | 24.69% | 28,354 t; refills 0–3; 0.15 per 1,000 t; median year 2005; 17.8% temperature regulated |
| 4 | High volume — frequently refilled | 30.61% | 28,864 t; refills 4–8; 0.15 per 1,000 t; median year 2005; 42.2% temperature regulated |

**Objective 1, answered.** Per ton shipped, the **high-volume tier is the strongest** part of the network and the
**transport-problem segment the weakest**. Of seventeen recorded conditions, only three go with the differences:
**warehouse age** (older warehouses sit in higher-volume tiers), **certificate grade** (a small, consistent rise in A+
with volume), and **temperature regulation** (which separates the two high-volume sub-segments). **No recorded condition
explains the transport-problem segment.** Location, zone, capacity, ownership, infrastructure back-up, flood exposure,
staffing and market conditions do not differ between segments. The evidence shows that age matters
and location and ownership are weak.

**Limits.** Separation is weak; the 3/4 split is specific to K-Means; age findings rest on recorded years (52.5%) from
warehouses with three or more refill requests. **The data is a single snapshot, so all of this is association, not
cause.**

**Outputs:** `evaluation_metrics.csv`, `segment_separation.csv`,
`segment_performance_profile.csv`, `segment_condition_tests.csv`, `segment_condition_shares.csv`,
`segment_numeric_conditions.csv`, `segment_names.csv`, and four figures, all in `training_and_evaluation/`.

**Objective 1 is complete.**